### Домашнє завдання: Пониження розмірностей для Аналізу Портретів Клієнтів

#### Контекст
В цьому ДЗ ми попрацюємо з методами пониження розмірності на наборі даних для задачі аналізу портретів клієнтів (Customer Personality Analysis). **В попередньому ДЗ ми працювали з цими даними використовуючи кластеризацію, зараз використаємо кластеризацію і візуалізауємо результати з різними методами.**

Customer Personality Analysis - це аналіз різних сегментів клієнтів компанії. Цей аналіз дозволяє бізнесу краще розуміти своїх клієнтів і полегшує процес адаптації продуктів під конкретні потреби, поведінку та інтереси різних типів клієнтів.

Аналіз портретів клієнтів допомагає бізнесу змінювати свій продукт на основі цільової аудиторії, розділеної на різні сегменти. Наприклад, замість того, щоб витрачати гроші на маркетинг нового продукту для всіх клієнтів у базі даних компанії, бізнес може проаналізувати, який сегмент клієнтів найімовірніше придбає продукт, і потім зосередити маркетингові зусилля лише на цьому сегменті.

#### Вхідні дані
Вам надано набір даних з такими атрибутами:

**Характеристики користувачів:**
- `ID`: Унікальний ідентифікатор клієнта
- `Year_Birth`: Рік народження клієнта
- `Education`: Рівень освіти клієнта
- `Marital_Status`: Сімейний стан клієнта
- `Income`: Річний дохід домогосподарства клієнта
- `Kidhome`: Кількість дітей у домогосподарстві клієнта
- `Teenhome`: Кількість підлітків у домогосподарстві клієнта
- `Dt_Customer`: Дата реєстрації клієнта у компанії
- `Recency`: Кількість днів з моменту останньої покупки клієнта
- `Complain`: 1, якщо клієнт скаржився за останні 2 роки, 0 - якщо ні

**Продукти:**
- `MntWines`: Сума, витрачена на вино за останні 2 роки
- `MntFruits`: Сума, витрачена на фрукти за останні 2 роки
- `MntMeatProducts`: Сума, витрачена на м'ясні продукти за останні 2 роки
- `MntFishProducts`: Сума, витрачена на рибні продукти за останні 2 роки
- `MntSweetProducts`: Сума, витрачена на солодощі за останні 2 роки
- `MntGoldProds`: Сума, витрачена на золото за останні 2 роки

**Акції:**
- `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
- `AcceptedCmp1`: 1, якщо клієнт прийняв пропозицію у першій кампанії, 0 - якщо ні
- `AcceptedCmp2`: 1, якщо клієнт прийняв пропозицію у другій кампанії, 0 - якщо ні
- `AcceptedCmp3`: 1, якщо клієнт прийняв пропозицію у третій кампанії, 0 - якщо ні
- `AcceptedCmp4`: 1, якщо клієнт прийняв пропозицію у четвертій кампанії, 0 - якщо ні
- `AcceptedCmp5`: 1, якщо клієнт прийняв пропозицію у п'ятій кампанії, 0 - якщо ні
- `Response`: 1, якщо клієнт прийняв пропозицію в останній кампанії, 0 - якщо ні

**Взаємодія з компанією:**
- `NumWebPurchases`: Кількість покупок, зроблених через вебсайт компанії
- `NumCatalogPurchases`: Кількість покупок, зроблених за каталогом
- `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
- `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць


Для початку, запустіть код нижче. Всі ці кроки ми робили в попередньому ДЗ і для того, щоб результати кластеризації у нас були схожими, потрібно аби передобробка була однаковою.

In [1]:
import pandas as pd

# 1. Завантаження даних
df = pd.read_csv('marketing_campaign.csv', sep='\t')

# 2. Обробка пропущених значень
df['Income_not_filled'] = df.Income.isna()
df.Income = df.Income.fillna(-1)

# 3. Обробка дати реєстрації
df.Dt_Customer = pd.to_datetime(df.Dt_Customer, format='%d-%m-%Y')
today = df.Dt_Customer.max()
df['days_lifetime'] = (today - df.Dt_Customer).dt.days
df['years_customer'] = df.Year_Birth.apply(lambda x: today.year - x)

# 4. Категоризація рівня освіти
df_education = pd.get_dummies(df.Education, prefix='education').astype(int)
df = pd.concat([df, df_education], axis=1)

# 5. Очищення сімейного стану
marital_status_map = {'Alone': 'Single', 'Absurd': 'Else', 'YOLO': 'Else'}
df['Marital_Status_clean'] = df.Marital_Status.map(marital_status_map)
df_ms = pd.get_dummies(df.Marital_Status_clean, prefix='marital').astype(int)
df = pd.concat([df, df_ms], axis=1)

# 6. Форматування доходу і видалення викиду
df.Income = df.Income.astype(int)
df = df[df.Income != 666666]

# 7. Створення фінального набору даних
X = df.drop(['ID', 'Dt_Customer', 'Education', 'Marital_Status', 'Marital_Status_clean'], axis=1)
X.reset_index(drop=True, inplace=True)

### Завдання 1: Виконання кластеризації та пониження розмірності для візуалізації результатів

Ваше завдання — провести кластеризацію клієнтів та візуалізувати результати кластеризації, використовуючи метод головних компонент (PCA) для пониження розмірності даних.

#### Інструкції:

1. **Вибір ключових характеристик:**
   Давайте обмежимось тільки наступними хараткеристиками для кластеризації цього разу:
   - `Income`: Річний дохід домогосподарства клієнта
   - `Recency`: Кількість днів з моменту останньої покупки клієнта
   - `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
   - `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
   - `days_lifetime`: Кількість днів з моменту реєстрації клієнта у компанії
   - `years_customer`: Вік клієнта
   - `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць
   Відберіть в наборі даних `X` лише ці характеристики.

2. **Стандартизація даних:**
   Використайте метод `StandardScaler` для стандартизації значень обраних характеристик.
   
   **Чому не MinMaxScaler:**
   - Для PCA краще використовувати StandardScaler, бо він вирівнює дисперсію ознак, на відміну від MinMaxScaler, що просто масштабує значення без врахування варіації.

   - Для K-Means також краще використовувати StandardScaler, бо алгоритм чутливий до масштабів: фічі з більшими значеннями сильніше впливають на обчислення відстаней, що може спотворити кластери.

3. **Кластеризація:**
   Проведіть кластеризацію клієнтів, використовуючи метод `KMeans` з трьома кластерами.

4. **Пониження розмірності:**
   Використайте метод головних компонент (PCA) для пониження розмірності даних до трьох компонент.

5. **Візуалізація результатів:**
   Використовуючи plolty express побудуйте 3D-графік розподілу клієнтів у просторі трьох головних компонент, де кольором позначено кластери.

6. **Опишіть, що спостерігаєте:**
   Чи кластеризація чітко розділила дані?

Далі ми детальніше проінтерпретуємо результати візуалізації і пониження розмірностей.

In [2]:
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Відбір ключових характеристик
features = ['Income', 'Recency', 'NumStorePurchases', 'NumDealsPurchases',
            'days_lifetime', 'years_customer', 'NumWebVisitsMonth']
X_sel = X[features].copy()

# 2. Стандартизація
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# 3. Кластеризація KMeans (3 кластери)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
print('Розмір кластерів:')
print(pd.Series(clusters).value_counts().sort_index())

# 4. PCA до 3 компонент
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])
df_pca['cluster'] = clusters.astype(str)
df_pca = pd.concat([df_pca, X_sel.reset_index(drop=True)], axis=1)

# 5. 3D-візуалізація
fig = px.scatter_3d(
    df_pca, x='PC1', y='PC2', z='PC3', color='cluster',
    hover_data=features, opacity=0.7,
    category_orders={'cluster': ['0', '1', '2']},
    title='Кластери клієнтів у просторі трьох головних компонент (PCA)'
)
fig.update_traces(marker=dict(size=3))
fig.show()

Розмір кластерів:
0    961
1    823
2    455
Name: count, dtype: int64


**Спостереження (Завдання 1):**

- KMeans розбив клієнтів на 3 групи: ~960, ~820 і ~455 клієнтів.
- У просторі PC1–PC2 кластери розділені досить добре: кластер 0 лежить зліва (низький PC1), кластер 1 — справа (високий PC1), кластер 2 — вгорі (високий PC2). Межі між ними не різкі, кластери дотикаються, а не утворюють окремі «острови». Дані неперервні, і KMeans ділить одну хмару точок на частини.
- Вздовж PC3 кластери майже не відрізняються: усі три розтягнуті по цій осі однаково. Отже, PC3 описує ознаку, яка не впливала на поділ на кластери (у Завданні 3 видно, що це `Recency`).
- Висновок: кластеризація розділила дані змістовно, але не «чисто». Розділення помітне в площині PC1–PC2, а перекриття є на межах.

### Завдання 2: Аналіз результатів пониження розмірності

1. **Розрахунок частки поясненої дисперсії:**
   Визначте, яка частка загальної варіації даних пояснюється кожною з трьох головних компонент (PC1, PC2, PC3) за допомогою атрибуту `explained_variance_ratio_` об'єкта PCA. Виведіть результат на екран.

2. **Розрахунок кумулятивної частки поясненої дисперсії:**
   Обчисліть кумулятивну частку поясненої дисперсії для трьох головних компонент, щоб зрозуміти, скільки варіації даних пояснюється першими кількома компонентами.

In [3]:
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

variance_df = pd.DataFrame({
    'Explained variance ratio': explained.round(4),
    'Cumulative': cumulative.round(4)
}, index=['PC1', 'PC2', 'PC3'])
print(variance_df)

fig = px.bar(variance_df.reset_index(), x='index', y='Explained variance ratio',
             title='Частка поясненої дисперсії по компонентах',
             labels={'index': 'Компонента'}, text_auto='.1%')
fig.add_scatter(x=variance_df.index, y=variance_df['Cumulative'],
                mode='lines+markers', name='Кумулятивна')
fig.show()

     Explained variance ratio  Cumulative
PC1                    0.3185      0.3185
PC2                    0.1966      0.5151
PC3                    0.1434      0.6585


**Висновок (Завдання 2):**

- PC1 пояснює ≈ **31.8%** дисперсії, PC2 ≈ **19.7%**, PC3 ≈ **14.3%**.
- Разом три компоненти пояснюють ≈ **65.8%** загальної варіації.
- Отже, 3D-графік передає приблизно дві третини інформації з 7 ознак, а третина губиться. Це нормально для візуалізації, але кластери, які на графіку перекриваються, у повному 7-вимірному просторі можуть бути розділені краще.
- Варіація розподілена між компонентами досить рівномірно (немає однієї компоненти на 90%). Це означає, що ознаки слабко корелюють між собою і кожна несе власну інформацію.

### Завдання 3: Інтерпретація "Loadings"

Продовжуємо інтерпретацію результатів `PCA`і познайомимось з новим поняттям `loadings`, яке допоможе нам знайти звʼязок між головними компонентами і оригінальними ознаками в наборі даних.

Ми зараз побудували візуалізацію кластерів точок даних в просторі трьох головних компонент. Але хочеться знайти звʼязок між головними компонентами і оригінальними ознаками. Для розуміння, які початкові характеристики даних мають найбільший вплив на ці головні компоненти, ми можемо використати атрибут `components_` методу `PCA`.

#### Що таке `pca.components_`?

`pca.components_` — це масив, який містить коефіцієнти (або "ваги"), що показують внесок кожної вихідної ознаки у кожну з головних компонент. Ці коефіцієнти ще називаються **"loading"** або "навантаженнями" компонент.

- **Loadings** (`навантаження`) відображають важливість кожної змінної (ознаки) для відповідної головної компоненти. Вони показують, яким чином змінні поєднуються, щоб утворити нові, зменшені вимірювання.
- Якщо коефіцієнт має високе абсолютне значення (як позитивне, так і негативне), це вказує на те, що відповідна змінна сильно впливає на головну компоненту.

#### Саме завдання
Ваше завдання — обчислити "навантаження" для кожної з головних компонент і інтерпретувати результати.

1. **Обчислення loadings для компонент:**
   Використайте атрибут `components_` об'єкта PCA для створення DataFrame, який відображатиме внесок кожної вихідної ознаки в кожну головну компоненту.

2. **Інтерпретація результатів:**
   Виведіть значення "навантажень" і проаналізуйте, які ознаки найбільше впливають на кожну головну компоненту.

In [4]:
loadings = pd.DataFrame(pca.components_.T, index=features, columns=['PC1', 'PC2', 'PC3'])
display(loadings.round(3).style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1))

# Топ-ознаки за абсолютним навантаженням для кожної компоненти
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(3).index
    print(f'{pc}:', ', '.join(f'{f} ({loadings.loc[f, pc]:+.2f})' for f in top))

# Середні значення компонент по кластерах — звʼязок кластерів з компонентами
print('\nСередні координати кластерів у просторі PCA:')
print(df_pca.groupby('cluster')[['PC1', 'PC2', 'PC3']].mean().round(2))

,PC1,PC2,PC3
Income,0.584000,0.165000,-0.044000
Recency,0.010000,0.037000,0.976000
NumStorePurchases,0.489000,0.385000,-0.081000
NumDealsPurchases,-0.199000,0.608000,-0.035000
days_lifetime,-0.133000,0.603000,-0.022000
years_customer,0.189000,0.209000,0.193000
NumWebVisitsMonth,-0.572000,0.216000,-0.016000


PC1: Income (+0.58), NumWebVisitsMonth (-0.57), NumStorePurchases (+0.49)
PC2: NumDealsPurchases (+0.61), days_lifetime (+0.60), NumStorePurchases (+0.38)
PC3: Recency (+0.98), years_customer (+0.19), NumStorePurchases (-0.08)

Середні координати кластерів у просторі PCA:
          PC1   PC2   PC3
cluster                  
0       -1.13 -0.68  0.03
1        1.62 -0.09 -0.03
2       -0.54  1.61 -0.02


**Інтерпретація loadings (Завдання 3):**

- **PC1 — «платоспроможність / офлайн- vs онлайн-поведінка».** Найбільші ваги: `Income` (+0.58), `NumWebVisitsMonth` (−0.57), `NumStorePurchases` (+0.49). Високий PC1 мають заможні клієнти, які багато купують у магазинах і рідко заходять на сайт. Низький PC1 — клієнти з меншим доходом, які часто відвідують сайт.
- **PC2 — «лояльність і чутливість до знижок».** Найбільші ваги: `NumDealsPurchases` (+0.61), `days_lifetime` (+0.60), `NumStorePurchases` (+0.39). Високий PC2 — давні клієнти, які активно купують зі знижками.
- **PC3 — майже повністю `Recency`** (+0.98). Ця компонента описує лише давність останньої покупки й не повʼязана з іншими ознаками.
- `years_customer` (вік) має помірні ваги на всіх трьох компонентах і не домінує в жодній.

**Звʼязок із кластерами:**
- Кластер 1 має високий PC1 (≈ +1.6): заможні покупці в магазинах.
- Кластер 0 має низький PC1 (≈ −1.1) і PC2 (≈ −0.7): молодші клієнти з невисоким доходом, нові й активні на сайті.
- Кластер 2 має високий PC2 (≈ +1.6): давні клієнти, які полюють на знижки.
- По PC3 (`Recency`) усі кластери мають середнє ≈ 0. Це пояснює, чому на 3D-графіку вони не розділяються вздовж цієї осі.

###Завдання 4
Давайте проаналізуємо "навантаження" (**loadings**) для трьох головних компонент після вилучення ознаки `Income`. Це допоможе нам зрозуміти, як змінилася важливість інших ознак для кожної головної компоненти, коли одна з ключових характеристик (`Income`) була вилучена.

#### Кроки для проведення аналізу і ваше завдання:

1. Видаліть ознаку `Income` з нашого набору даних `X` і повторно виконайте PCA (метод головних компонент) для отримання нових "навантажень".

2. Обчисліть нові "навантаження" для трьох головних компонент на наборі даних без `Income`

3. Проаналізуйте, які ознаки мають найбільший вплив на кожну головну компоненту після вилучення `Income`.

4. Перегляньте, наскільки кожна з головних компонент пояснює дисперсію в даних без ознаки `Income`.

In [5]:
features_no_income = [f for f in features if f != 'Income']
X_no_income = X[features_no_income]

scaler_ni = StandardScaler()
X_ni_scaled = scaler_ni.fit_transform(X_no_income)

pca_ni = PCA(n_components=3, random_state=42)
X_ni_pca = pca_ni.fit_transform(X_ni_scaled)

loadings_ni = pd.DataFrame(pca_ni.components_.T, index=features_no_income,
                           columns=['PC1', 'PC2', 'PC3'])
display(loadings_ni.round(3).style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1))

for pc in loadings_ni.columns:
    top = loadings_ni[pc].abs().sort_values(ascending=False).head(3).index
    print(f'{pc}:', ', '.join(f'{f} ({loadings_ni.loc[f, pc]:+.2f})' for f in top))

variance_cmp = pd.DataFrame({
    'З Income': pca.explained_variance_ratio_,
    'Без Income': pca_ni.explained_variance_ratio_,
}, index=['PC1', 'PC2', 'PC3'])
variance_cmp.loc['Сума'] = variance_cmp.sum()
print('\nЧастка поясненої дисперсії:')
print(variance_cmp.round(4))

,PC1,PC2,PC3
Recency,-0.014000,0.075000,0.990000
NumStorePurchases,-0.381000,0.629000,-0.111000
NumDealsPurchases,0.446000,0.462000,-0.062000
days_lifetime,0.384000,0.468000,0.007000
years_customer,-0.177000,0.394000,0.056000
NumWebVisitsMonth,0.691000,-0.108000,0.010000


PC1: NumWebVisitsMonth (+0.69), NumDealsPurchases (+0.45), days_lifetime (+0.38)
PC2: NumStorePurchases (+0.63), days_lifetime (+0.47), NumDealsPurchases (+0.46)
PC3: Recency (+0.99), NumStorePurchases (-0.11), NumDealsPurchases (-0.06)

Частка поясненої дисперсії:
      З Income  Без Income
PC1     0.3185      0.2739
PC2     0.1966      0.2144
PC3     0.1434      0.1668
Сума    0.6585      0.6552


**Аналіз (Завдання 4):**

Знак компоненти в PCA довільний: якщо всі ваги компоненти помножити на −1, зміст не зміниться. Тому дивимось на абсолютні значення й на те, які ознаки мають однаковий знак.

- **PC1 без Income:** `NumWebVisitsMonth` (+0.69), `NumDealsPurchases` (+0.45), `days_lifetime` (+0.38), а `NumStorePurchases` (−0.38) має протилежний знак. Без доходу головною віссю стала **онлайн-активність**: часті відвідування сайту й покупки зі знижками протиставлені покупкам у магазині. Раніше PC1 визначав переважно `Income`, тепер його місце зайняв `NumWebVisitsMonth`, який сильно (негативно) корелював із доходом.
- **PC2 без Income:** `NumStorePurchases` (+0.63), `days_lifetime` (+0.47), `NumDealsPurchases` (+0.46), `years_customer` (+0.39). Це «загальна активність і стаж»: давні, старші клієнти, які багато купують. Вага віку тут помітно зросла, бо вік теж корелював із доходом.
- **PC3** не змінився: це так само майже чистий `Recency` (+0.99).

**Пояснена дисперсія:**
- З Income: 31.8% / 19.7% / 14.3% → разом **65.8%**.
- Без Income: 27.4% / 21.4% / 16.7% → разом **65.5%**.
- PC1 «схуд» на ~4.5 п.п., бо зникла ознака, яка разом з `NumStorePurchases` і `NumWebVisitsMonth` утворювала найсильніший корельований блок. Дисперсія розподілилась між компонентами рівномірніше.
- Сумарно три компоненти пояснюють майже стільки ж, бо даних стало менше на одну ознаку і на їх частку припадає більша доля. Але **структура** першої компоненти змінилась: `Income` був ключовою ознакою, що «склеювала» перший фактор (платоспроможність).

### Завдання 5: Візуалізація кластеризації за допомогою t-SNE

Ваше завдання — використати метод t-SNE для візуалізації результатів кластеризації клієнтів у двовимірному просторі. Метод t-SNE допомагає знизити розмірність даних та зберегти локальні структури в даних, що робить його ефективним для візуалізації високорозмірних даних. Ми також зможемо порівняти результат цього методу з РСА.

1. Використайте метод t-SNE для зниження розмірності до 2х вимірів даних, які включають ознаки всі, що і в завданні 1, а також були відмасштабовані перед пониженням розмірностей.

2. Створіть новий DataFrame з координатами, отриманими після застосування t-SNE, та додайте до нього мітки кластерів.

3. Побудуйте інтерактивний 2D-графік розподілу клієнтів, де кольором буде позначено різні кластери і проаналізуйте графік з рекомендаціями нижче (можливо треба буде вивести додаткові візуалізації чи таблиці для інтерпретації, але треба прям зрозуміти, які ознаки формують який кластер і чим кластери відрізняються одне від одного).

  **Опишіть отримані кластери з точки зору ознак.**

4. Опишіть відмінність графіка tSNE від PCA.

#### ЯК можна інтерпретувати з t-SNE?

Хоча t-SNE не надає "компонентів" як РСА, він забезпечує низьковимірне представлення даних, яке можна візуально інтерпретувати:

- **Кластери:** t-SNE особливо добре показує кластери подібних точок. Якщо ви бачите чітко визначені кластери на графіку t-SNE, це свідчить про наявність груп схожих спостережень у ваших даних. Проаналізувати їх можемо, якщо додамо дані в `hover_data` або якщо якісь з даних виведемо як розмір чи форма точок на візуалізації. Також корисно може бути вивести середні значення ознак по кластерам.
- **Локальна структура:** Відносне розташування точок одного кластеру на графіку t-SNE може допомогти вам зрозуміти, які дані подібні між собою.
- **Глобальна структура:** Будьте обережні; t-SNE менш надійний для відображення глобальних структур (наприклад, відстаней між кластерами) у порівнянні з PCA, бо t-SNE націлений на збереження саме локальних структур.

In [6]:
from sklearn.manifold import TSNE

# 1. t-SNE на тих самих відмасштабованих ознаках, що й у Завданні 1
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca')
X_tsne = tsne.fit_transform(X_scaled)

# 2. DataFrame з координатами та мітками кластерів
df_tsne = pd.DataFrame(X_tsne, columns=['TSNE1', 'TSNE2'])
df_tsne['cluster'] = clusters.astype(str)
df_tsne = pd.concat([df_tsne, X_sel.reset_index(drop=True)], axis=1)

# 3. Інтерактивний 2D-графік
fig = px.scatter(
    df_tsne, x='TSNE1', y='TSNE2', color='cluster',
    hover_data=features, opacity=0.7,
    category_orders={'cluster': ['0', '1', '2']},
    title='Кластери клієнтів після t-SNE (2D)'
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [7]:
# Середні значення ознак по кластерах — для інтерпретації
cluster_profile = df_tsne.groupby('cluster')[features].mean()
cluster_profile['size'] = df_tsne['cluster'].value_counts().sort_index()
display(cluster_profile.round(1))

# Відхилення від загального середнього в стандартних відхиленнях — зручно порівнювати ознаки
z_profile = pd.DataFrame(X_scaled, columns=features).groupby(clusters).mean()
z_profile.index = z_profile.index.astype(str)
fig = px.imshow(z_profile.T.round(2), text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1.5, zmax=1.5, aspect='auto',
                labels=dict(x='Кластер', y='Ознака', color='z-score'),
                title='Профіль кластерів (середнє стандартизоване значення ознаки)')
fig.show()

,Income,Recency,NumStorePurchases,NumDealsPurchases,days_lifetime,years_customer,NumWebVisitsMonth,size
cluster,,,,,,,,
0,33054.9,48.5,3.2,1.8,312.6,41.9,6.5,961
1,72551.6,49.3,8.3,1.4,315.3,47.3,3.1,823
2,51950.7,50.1,6.7,5.0,509.5,48.2,6.9,455


In [8]:
# Ті самі t-SNE координати, забарвлені за окремими ознаками — видно, які ознаки формують розташування точок
for feat in ['Income', 'NumDealsPurchases', 'days_lifetime', 'Recency']:
    fig = px.scatter(df_tsne, x='TSNE1', y='TSNE2', color=feat,
                     color_continuous_scale='Viridis', opacity=0.7,
                     title=f't-SNE, колір = {feat}')
    fig.update_traces(marker=dict(size=4))
    fig.show()

**Опис кластерів (Завдання 5):**

| Кластер | Розмір | Портрет |
|---|---|---|
| **0 — «Молоді економні онлайн-відвідувачі»** | ~960 | Найнижчий дохід (~33 тис.), наймолодші (~42 роки), мало покупок у магазині (~3), мало покупок зі знижкою (~1.8), часто заходять на сайт (~6.5 разів/міс), відносно нові клієнти (~313 днів). |
| **1 — «Заможні покупці в магазинах»** | ~820 | Найвищий дохід (~73 тис.), найбільше покупок у магазині (~8.3), найменше покупок зі знижкою (~1.4), рідко заходять на сайт (~3 рази/міс). Стаж такий самий, як у кластера 0 (~315 днів). |
| **2 — «Лояльні мисливці за знижками»** | ~455 | Середній дохід (~52 тис.), найстарші (~48 років), **найдовший стаж** (~510 днів), **найбільше покупок зі знижкою** (~5), часто заходять на сайт (~6.9) і багато купують у магазині (~6.7). |

- `Recency` майже однаковий у всіх кластерах (~49–50 днів), тож він **не формує** сегменти.
- Головні ознаки, що розділяють кластери: `Income`, `NumWebVisitsMonth`, `NumStorePurchases` (кластер 0 vs 1) та `NumDealsPurchases`, `days_lifetime` (кластер 2 vs решта).

**Практичні рекомендації:**
- Кластер 1: преміальні товари й офлайн-акції в магазинах. Знижки для них мало важливі.
- Кластер 2: програми лояльності й персональні знижки. Вони реагують на акції і давно з компанією.
- Кластер 0: онлайн-маркетинг (сайт уже відвідують) і недорогі пропозиції для конверсії відвідувань у покупки.

**Чим t-SNE відрізняється від PCA:**
- **PCA** — лінійна проєкція, яка зберігає глобальну структуру. Осі мають зміст (через loadings), а відстані між точками можна інтерпретувати. На графіку PC1–PC2 видно щільне ядро з кількома віддаленими викидами (клієнти з незаповненим доходом `Income = -1` та нереалістичним віком ~115–121 рік). Кластери ділять ядро на три сектори: 0 ліворуч, 1 праворуч, 2 зверху.
- **t-SNE** — нелінійний метод, що зберігає **локальну** структуру (хто з ким сусід). Він «розгортає» щільне ядро на всю площину, тож точки розподілені рівномірніше, а викиди не стискають решту даних. Кластери, як і в PCA, займають суміжні області (0 ліворуч-знизу, 1 праворуч, 2 зверху) без чітких проміжків між ними. Це підтверджує, що в даних немає ізольованих груп: KMeans ділить неперервну хмару. Якщо розфарбувати t-SNE за `Recency`, видно дрібні локальні «плями» з однаковою давністю покупки всередині кожного кластера. t-SNE зберіг цю локальну схожість, хоча на поділ на кластери вона не вплинула.
- Осі t-SNE **не мають змісту**, а розміри кластерів і відстані між ними на графіку не відповідають реальним. Тому для інтерпретації ознак використовуємо PCA-loadings та середні значення по кластерах, а t-SNE — для візуальної перевірки, чи є в даних групи.
- t-SNE стохастичний (результат залежить від `random_state` і `perplexity`) і працює значно повільніше за PCA.